In [2]:
import pandas as pd
from datasets import load_dataset
from transform_tools import chunk, clean_text
from numpy import random

/Users/andrii/Developer/nure/which-movie-was-that/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:


df = load_dataset("Coder-Dragon/wikipedia-movies", split="train").to_pandas()
df.shape

(30671, 9)

In [4]:

df = df[df['Origin/Ethnicity'] == 'American']
df = df[df['Release Year'] > 1950]
df.shape

(10951, 9)

In [5]:
df = df[df['Plot'].str.len() > 100]
df['Plot_clean'] = df['Plot'].apply(clean_text)
df.shape

(10918, 10)

In [6]:
chunked_data = []

In [7]:
import uuid

chunk_max_size = 350

for idx, plot, wiki_page in zip(df.index, df['Plot_clean'], df['Wiki Page']):                     
    for chunk_no, chunk_text in enumerate(chunk(plot, chunk_max_size)):                                      
        chunked_data.append({                                                                     
            'uuid': str(uuid.uuid4()),                           
            'doc_id': wiki_page,                                      
            'chunk_index_in_document': chunk_no,                        
            'text': chunk_text,                                                                  
        })

In [ ]:
# pd.DataFrame(chunked_data).to_json('chunked_data.jsonl', orient='records', lines=True)

In [9]:
total_chunks = len(chunked_data)
average_chunk_size = sum(len(chunk['text'].split()) for chunk in chunked_data) / total_chunks
median_chunk_size = sorted(len(chunk['text'].split()) for chunk in chunked_data)[total_chunks // 2]

total_chunks, average_chunk_size, median_chunk_size

(21892, 264.02343321761373, 350)